This file is designed to test sign language detection models in real-time by capturing video input from a camera. When executed, the script activates the camera, allowing users to perform sign language gestures while the models make real-time predictions. This interactive setup enables immediate feedback on the accuracy and responsiveness of the trained models, providing a practical demonstration of how they perform in a live environment. It serves as an effective tool for validating model performance and can be used for further fine-tuning or user demonstrations.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from keras.models import load_model

model = load_model("../trained-models/resnet_landmarks_model.h5")

cap = cv2.VideoCapture(0)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3, max_num_hands=2)

labels_dict = {i: chr(65 + i) for i in range(26)}


def get_best_hand_details(results, labels_dict):
    highest_confidence = 0
    best_hand_landmarks = None
    best_predicted_character = None

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            if results.multi_handedness:
                handedness = results.multi_handedness[0].classification[0].label
                if handedness == "Left":
                    data_aux = []
                    for i in range(len(hand_landmarks.landmark)):
                        x = hand_landmarks.landmark[i].x
                        y = hand_landmarks.landmark[i].y
                        z = hand_landmarks.landmark[i].z  # Assuming you also have z coordinate

                        data_aux.extend([x, y, z])

              
                   
                    data_aux = np.array(data_aux).reshape((1, 21, 3))  # Assuming the model expects sequences of 21 timesteps with 3 features each

                    # Make prediction
                    prediction = model.predict(data_aux)

                    # Get predicted character and confidence
                    predicted_character = np.argmax(prediction)
                    confidence = prediction[0][predicted_character]

                    if confidence > highest_confidence:
                        highest_confidence = confidence
                        best_hand_landmarks = hand_landmarks
                        best_predicted_character = labels_dict[predicted_character]

    return best_hand_landmarks, best_predicted_character, highest_confidence

while True:
    ret, frame = cap.read()

    if not ret:
        # If the frame is not successfully captured, break out of the loop
        break

    H, W, _ = frame.shape

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(frame_rgb)

    best_hand_landmarks, best_predicted_character, highest_confidence = get_best_hand_details(results, labels_dict)

    if best_hand_landmarks is not None and highest_confidence >= 0.5:  # Check if confidence is >= 50%
        # Determine if the hand is left or right
        mp_drawing.draw_landmarks(
            frame,  # image to draw
            best_hand_landmarks,  # model output
            mp_hands.HAND_CONNECTIONS,  # hand connections
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())

        # Get bounding box coordinates
        x1 = int(min(hand_landmark.x * W for hand_landmark in best_hand_landmarks.landmark))
        y1 = int(min(hand_landmark.y * H for hand_landmark in best_hand_landmarks.landmark))
        x2 = int(max(hand_landmark.x * W for hand_landmark in best_hand_landmarks.landmark))
        y2 = int(max(hand_landmark.y * H for hand_landmark in best_hand_landmarks.landmark))

        # Draw rectangle around the hand
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Draw predicted letter above the hand
        cv2.putText(frame, f'{best_predicted_character} ({highest_confidence:.2f}%)',
                    (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()